# Notebook 7 — Train / Validation / Test Split

**Goals**

1. Understand why **random** splits are wrong for time series.
2. Use four time-aware splitting strategies:
   - Single chronological split
   - Holdout-horizon split (last *N* periods as test)
   - Expanding-window walk-forward
   - Rolling-window walk-forward
3. Visualise each split.
4. Verify with `assert_no_leakage` that no train row is dated after a
   validation/test row.


In [1]:
# ──────────────────────────────────────────────────────────────────────────
# COLAB SETUP — run this once at the top of every tutorial notebook.
# It installs plotly + statsmodels and makes the toolkit importable.
# If you are running locally (not in Colab) the !pip line is harmless.
# ──────────────────────────────────────────────────────────────────────────
!pip install -q plotly statsmodels
import sys, os
# If you uploaded forecasting_toolkit.zip to Colab, unzip it once:
#   !unzip -o forecasting_toolkit.zip
# Otherwise place forecasting_toolkit/ next to this notebook.
sys.path.insert(0, os.path.abspath('.'))

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = 'colab'   # change to 'notebook' for local Jupyter


In [2]:
# ──────────────────────────────────────────────────────────────────────────
# POINT THIS AT YOUR DATASET — fill in the four lines below.
# Everything in this notebook works for ANY tabular sales/demand dataset
# (M5, Rossmann, Walmart Store Item Demand, custom CSVs, etc.).
#
#   DATA_PATH    – path to your CSV / Parquet file
#   DATE_COL     – name of the timestamp column
#   TARGET_COL   – name of the column you want to forecast
#   KEY_COLS     – list of columns that together identify ONE time series
#   STATIC_COLS  – columns constant within a key (e.g. store_type, category)
#   DYNAMIC_COLS – columns that vary in time within a key (e.g. promo, price)
#   FREQUENCY    – pandas offset alias: 'D','W','MS','H',…
# ──────────────────────────────────────────────────────────────────────────
DATA_PATH    = './datasets/rohlik_kaggle/sales_processed_tft.csv'
DATE_COL     = 'date'
TARGET_COL   = 'sales'
KEY_COLS     = ['unique_id']
STATIC_COLS  = ['warehouse','product_unique_id','name','L1_category_name_en','L2_category_name_en','L3_category_name_en','L4_category_name_en','country']
DYNAMIC_COLS = ['total_orders','sell_price_main','type_0_discount','type_1_discount','type_2_discount','type_3_discount','type_4_discount','type_5_discount','type_6_discount','holiday_name',
                'holiday','shops_closed','winter_school_holidays','school_holidays','weekday','week','month','day','is_month_start','is_month_end','quarter','weekend','days_since_2020']
FREQUENCY    = 'D'

from forecasting_toolkit import data_io
spec = data_io.make_spec(
    date_col=DATE_COL, target_col=TARGET_COL,
    key_cols=KEY_COLS, static_cols=STATIC_COLS,
    dynamic_cols=DYNAMIC_COLS, frequency=FREQUENCY,
)
df = data_io.load_data(DATA_PATH, spec)
print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
df.head()


Loaded 4,054,440 rows × 38 columns


,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,...,month,year,prev_year,day,is_month_start,is_month_end,quarter,weekend,days_since_2020,country
0,0,2022-07-18,Budapest_1,5289.0,3.97,710.89,0.09,0.0,0.0,0.00000,...,7,2022,2022,18,False,False,3,0,929,Hungary
1,0,2022-07-19,Budapest_1,5255.0,73.36,710.89,1.00,0.0,0.0,0.00000,...,7,2022,2022,19,False,False,3,0,930,Hungary
2,0,2022-07-20,Budapest_1,5334.0,558.09,710.89,0.96,0.0,0.0,0.45045,...,7,2022,2022,20,False,False,3,0,931,Hungary
3,0,2022-07-21,Budapest_1,5459.0,14.03,710.89,0.06,0.0,0.0,0.45045,...,7,2022,2022,21,False,False,3,0,932,Hungary
4,0,2022-07-22,Budapest_1,5461.0,558.53,710.89,0.97,0.0,0.0,0.45045,...,7,2022,2022,22,False,False,3,0,933,Hungary


In [3]:
import forecasting_toolkit as ft
import pandas as pd
print(f'Date range: {df[DATE_COL].min()} → {df[DATE_COL].max()}')
print(f'Total rows: {len(df):,}')
print(f'Total keys: {df[KEY_COLS].drop_duplicates().shape[0]:,}')


Date range: 2020-08-01 00:00:00 → 2024-06-16 00:00:00
Total rows: 4,054,440
Total keys: 5,390


## 7.1 Why **not** a random split?

The default `train_test_split` shuffles rows uniformly at random. For
i.i.d. tabular data that's fine. For time series it produces **two
catastrophic problems**:

1. **Lookahead leakage.** A randomly chosen training row may be dated
   *after* a validation row. The model will use future data to predict
   the past, look amazing in evaluation, and fall over in production.
2. **Within-series leakage.** Even if you split by row index, a row at
   *t = 10* and a row at *t = 11* are essentially the same observation
   — the model isn't tested on truly unseen data.

**The rule:** every observation in the train set must be dated **strictly
before** every observation in the validation set, which must be dated
strictly before every observation in the test set.

```text
   train  ────────►│
                   │  val  ──────►│
                                  │  test  ──────►
```

The toolkit's splitters enforce this for you.


## 7.2 Single chronological split — `time_based_split`

You provide explicit cut-off dates. Most useful for fixed evaluation
windows specified by the business ("evaluate model accuracy on Q4 2023").


In [4]:
date_min = df[DATE_COL].min()
date_max = df[DATE_COL].max()
total_days = (date_max - date_min).days

# 70 / 15 / 15 split as a starting point
train_end = date_min + pd.Timedelta(days=int(total_days * 0.70))
val_end   = date_min + pd.Timedelta(days=int(total_days * 0.85))

split = ft.splitting.time_based_split(
    df, spec,
    train_end=train_end.strftime('%Y-%m-%d'),
    val_end  =val_end.strftime('%Y-%m-%d'),
)
display(split.summary())


,split,n_rows,pct,start,end_cutoff
0,train,2589420,63.87,0,2023-04-18
1,val,732888,18.08,0,2023-11-16
2,test,732132,18.06,1,2024-06-16


In [5]:
# Visualise on a few sample series
fig = ft.plotting.plot_split(split, spec, sample_keys=3,
        title='Time-based 70/15/15 split — three sample series')
fig.show()


### Verify no leakage

`assert_no_leakage` raises if it finds a train row dated after a val
row, or a val row dated after a test row.


In [6]:
ft.splitting.assert_no_leakage(split, spec)
print(' No leakage in this split.')


 No leakage in this split.


## 7.3 Holdout-horizon split — `holdout_horizon_split`

Often you want to forecast a **fixed-length future horizon** (e.g. the
next 28 days). This splitter works backwards from the end of the data:
the last `test_horizon` periods are the test set, the previous
`val_horizon` periods are validation, and everything before is training.

This is the standard setup for the M-competitions and most production
backtests.


In [7]:
horizon_split = ft.splitting.holdout_horizon_split(
    df, spec,
    test_horizon=28,   # forecast the last 28 periods
    val_horizon=28,    # use the 28 before that as validation
)
display(horizon_split.summary())
fig = ft.plotting.plot_split(horizon_split, spec, sample_keys=3,
        title='Holdout-horizon split — last 28 periods as test')
fig.show()


,split,n_rows,pct,start,end_cutoff
0,train,3864488,95.31,0,2024-04-21
1,val,95612,2.36,1,2024-05-19
2,test,94340,2.33,1,2024-06-16


In [8]:
ft.splitting.assert_no_leakage(horizon_split, spec)
print(' No leakage in the holdout-horizon split.')


 No leakage in the holdout-horizon split.


## 7.4 Walk-forward (rolling backtest) — the gold standard

A **single** train/val/test split tells you how the model performed
once. A **walk-forward backtest** tells you how the model performs
*on average* across several historical evaluation windows. It's how
production teams measure realistic accuracy.

| Strategy            | Train window grows? | When to prefer                                            |
|---------------------|---------------------|-----------------------------------------------------------|
| **Expanding window**| Yes — keeps all history | You believe older history still helps the model.       |
| **Rolling window**  | No — fixed size     | Distant history is no longer representative; markets shift. |


### Expanding-window walk-forward — `expanding_window_split`


In [10]:
# Start with the first ~50% as training, validate the next 28 periods,
# step by 28 periods, do up to 4 folds.
init_end = date_min + pd.Timedelta(days=int(total_days * 0.50))

exp_folds = ft.splitting.expanding_window_split(
    df, spec,
    initial_train_end=init_end.strftime('%Y-%m-%d'),
    val_horizon=28,
    n_folds=4,
    step=28,
)
print(f'Built {len(exp_folds)} expanding folds.')
for i, fold in enumerate(exp_folds, 1):
    print(f'  fold {i}: train={len(fold.train):>6,}  val={len(fold.val):>5,}  '
          f'val window={fold.train_end.date()} → {fold.val_end.date()}')


Built 4 expanding folds.
  fold 1: train=1,675,184  val=85,842  val window=2022-07-09 → 2022-08-06
  fold 2: train=1,761,026  val=87,011  val window=2022-08-06 → 2022-09-03
  fold 3: train=1,848,037  val=87,299  val window=2022-09-03 → 2022-10-01
  fold 4: train=1,935,336  val=88,912  val window=2022-10-01 → 2022-10-29


In [11]:
fig = ft.plotting.plot_walk_forward(exp_folds, spec,
        title='Expanding-window walk-forward')
fig.show()


### Rolling-window walk-forward — `rolling_window_split`


In [12]:
roll_folds = ft.splitting.rolling_window_split(
    df, spec,
    train_size=180,    # use last 180 periods as training
    val_horizon=28,
    n_folds=4,
    step=28,
)
print(f'Built {len(roll_folds)} rolling folds.')
fig = ft.plotting.plot_walk_forward(roll_folds, spec,
        title='Rolling-window walk-forward (fixed 180-period training)')
fig.show()


Built 4 rolling folds.


### Verify each fold


In [13]:
for i, fold in enumerate(exp_folds + roll_folds, 1):
    if len(fold.train) and len(fold.val):
        ft.splitting.assert_no_leakage(fold, spec)
print(f' All {len(exp_folds) + len(roll_folds)} folds are leakage-free.')


 All 8 folds are leakage-free.


## 7.7 Picking split parameters in practice

| Decision                  | Rule of thumb                                                    |
|---------------------------|-------------------------------------------------------------------|
| Test horizon              | Match what you're forecasting in production (e.g. next 28 days). |
| Validation horizon        | Same as test horizon, so val accuracy estimates test accuracy.   |
| Number of walk-forward folds | At least 3–5; more if data is plentiful.                      |
| Train window (rolling)    | Long enough to cover at least 2 complete seasonal cycles.        |
| Step between folds        | Equal to validation horizon for non-overlapping folds.           |

> Walk-forward backtests give you a **distribution** of metric values, not
> a single number. Report mean ± std across folds — it's much more
> informative than a single accuracy point estimate.


## 7.8 Take-aways

- A **time-aware split** is non-negotiable for forecasting. Random
  splits are silently broken.
- For one-shot evaluation use `time_based_split` or `holdout_horizon_split`.
- For honest accuracy estimates, use walk-forward
  (`expanding_window_split` or `rolling_window_split`).
- Always run `assert_no_leakage` after splitting, especially in
  notebooks where it's easy to slice things by hand.
  